# SnowGuard - Phase 2: INT8 TFLite export

Inputs (Kaggle *Add Input*): `cristvollerei/rscd-dataset-1million` and the `snowguard-export-kit` dataset (code + trained model + split manifest). CPU is enough; no Internet needed.

Produces `phase2_export.zip`: float32 and INT8 TFLite models, `labels.json` with the val-tuned class bias, and `export_report.json` (INT8 accuracy drop, ice / fresh-snow recall, hazard recall).

In [ ]:
import os, sys, shutil

def find_dir(root, match, max_depth=7):
    """Breadth-first search that never lists RSCD's huge image folders."""
    skip = {'train', 'test_50k', 'vali_20k', '__pycache__'}
    level = [root]
    for _ in range(max_depth):
        nxt = []
        for d in level:
            if match(d):
                return d
            if os.path.basename(d) in skip:
                continue
            try:
                nxt += [e.path for e in os.scandir(d) if e.is_dir()]
            except OSError:
                pass
        level = nxt
    return None

KIT = find_dir('/kaggle/input', lambda d: os.path.isfile(os.path.join(d, 'snowguard_ml', 'export.py')))
assert KIT, 'snowguard-export-kit not attached (needs snowguard_ml/export.py)'
MODEL_DIR = find_dir(KIT, lambda d: os.path.isfile(os.path.join(d, 'snowguard_mobilenetv2.keras')))
assert MODEL_DIR, 'snowguard_mobilenetv2.keras not found in the kit'
DATA_ROOT = find_dir('/kaggle/input', lambda d: os.path.basename(d) == 'train' and os.path.isdir(os.path.join(d, 'ice')))
assert DATA_ROOT, 'RSCD train/ directory not found'

sys.path.insert(0, KIT)
print('KIT       =', KIT)
print('MODEL_DIR =', MODEL_DIR, sorted(os.listdir(MODEL_DIR)))
print('DATA_ROOT =', DATA_ROOT)

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
from snowguard_ml import export

OUT = '/kaggle/working/export'
export.main([
    '--model', os.path.join(MODEL_DIR, 'snowguard_mobilenetv2.keras'),
    '--manifest', os.path.join(MODEL_DIR, 'manifest.csv'),
    '--data-root', DATA_ROOT,
    '--out', OUT,
    '--calib-images', '1500',
])

In [ ]:
bundle = '/kaggle/working/phase2_export'
os.makedirs(bundle, exist_ok=True)
for f in sorted(os.listdir(OUT)):
    if f.endswith(('.tflite', '.json', '.png')):
        shutil.copy(os.path.join(OUT, f), bundle)
shutil.rmtree(os.path.join(OUT, 'saved_model'), ignore_errors=True)
print(shutil.make_archive(bundle, 'zip', bundle))
print(sorted(os.listdir(bundle)))